# Pipeline ETL Completo - Situação Final dos Alunos
## Portal de Dados Abertos do Recife

Este notebook implementa um pipeline ETL completo para extrair, transformar e carregar dados de situação final dos alunos da rede municipal do Recife.

### Processo:
1. **Extração**: Coleta dados do Portal de Dados Abertos do Recife
2. **Transformação**: Limpeza, padronização e enriquecimento dos dados
3. **Carregamento**: Criação de esquema dimensional e carga no banco de dados

In [ ]:
pip install pandas numpy requests sqlalchemy psycopg2-binary

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
from datetime import datetime
from pathlib import Path
from sqlalchemy import create_engine

## Extraindo os dados

In [ ]:
# Criando um dicionário com todos os caminhos dos arquivos/URLs
urls_csv = {
    "2024": "http://dados.recife.pe.gov.br/dataset/ce5168d4-d925-48f5-a193-03d4e0f587c7/resource/96f8a467-12b1-4340-b19c-281907fabaae/download/situacaofinal2024.csv",
    "2023": "http://dados.recife.pe.gov.br/dataset/ce5168d4-d925-48f5-a193-03d4e0f587c7/resource/854da2d7-c34b-457f-97b9-ba217d489621/download/situacaofinal2023.csv",
    "2022": "http://dados.recife.pe.gov.br/dataset/ce5168d4-d925-48f5-a193-03d4e0f587c7/resource/9e22fc25-716f-4454-8d95-998894b6ce01/download/situacaofinal2022.csv"
}

# Criar diretórios se não existirem
Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("data/final").mkdir(parents=True, exist_ok=True)

lista_dataframes = []  # Para armazenar todos os dataframes

for ano, url in urls_csv.items():
    try:
        print(f"Extraindo dados do ano {ano}...")
        
        # Faz a requisição HTTP
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        # Lê o CSV direto da resposta
        df = pd.read_csv(pd.io.BytesIO(response.content), encoding='utf-8', sep=';', decimal=',')
        
        # Adiciona coluna com o ano de referência
        df['ano_referencia'] = int(ano)
        
        lista_dataframes.append(df)
        print(f"✓ Extraídos {len(df)} registros do ano {ano}")
        
    except Exception as e:
        print(f"Erro ao extrair dados do ano {ano}: {e}")

# Concatenando todos os DFs
df = pd.concat(lista_dataframes, ignore_index=True)
print(f"Total de linhas no df unificado: {len(df)}")

## Transformação

In [ ]:
# Verificando o resultado final e as informações sobre os tipos de dados
display(df.head(5))
display(df.info())

### Verificando valores únicos em cada uma das colunas

In [ ]:
# Valores únicos por coluna
df.nunique(dropna=False)  # Usar o "dropna=False" faz com que ele leve em consideração valores nulos

### Verificando os valores nulos por coluna

In [ ]:
print("Valores ausentes por coluna:")
print(df.isnull().sum())

In [ ]:
# Limpeza básica dos dados
print("=== LIMPEZA DE DADOS ===")

# Remove duplicatas
inicial = len(df)
df = df.drop_duplicates()
print(f"Duplicatas removidas: {inicial - len(df)}")

# Limpa strings - converte para maiúscula e remove espaços
string_columns = df.select_dtypes(include=['object']).columns
for col in string_columns:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.upper()
        # Substitui valores vazios por NaN
        df[col] = df[col].replace(['NAN', 'NONE', 'NULL', ''], np.nan)

print("✓ Limpeza básica concluída")

In [ ]:
# Tratamento de valores nulos baseado na análise dos dados
# Remove linhas com muitos valores nulos (mais de 50% das colunas vazias)
threshold = len(df.columns) * 0.5
df_clean = df.dropna(thresh=threshold)

print(f"Linhas removidas por excesso de nulos: {len(df) - len(df_clean)}")

# Adiciona informações de processamento
df_clean['data_processamento'] = datetime.now()

# Adiciona ID sequencial
df_clean = df_clean.reset_index(drop=True)
df_clean.insert(0, 'id', range(1, len(df_clean) + 1))

print(f"Dataset após limpeza: {len(df_clean)} registros")
print("✓ Transformação concluída")

### Resultado final da transformação

In [ ]:
display(df_clean.info())
display(df_clean.head())

## Carregamento: Transformando em um esquema estrela e injetando o dataset no Banco de Dados PostgreSQL

### Criando o esquema estrela

Para transformar em um esquema estrela, vamos criar as dimensões baseadas nos dados dos alunos:

- **Dimensão Tempo** (quando ocorreu)
- **Dimensão Ano** (ano de referência dos dados)  
- **Dimensão Escola** (características da escola/unidade)
- **Dimensão Aluno** (características dos alunos)
- **Tabela Fato** (situação final dos alunos)

In [ ]:
# 1. Criando a Dimensão Tempo
datas_unicas = pd.DataFrame({'data_completa': [datetime.now().date()]})
datas_unicas['ano'] = pd.to_datetime(datas_unicas['data_completa']).dt.year
datas_unicas['mes'] = pd.to_datetime(datas_unicas['data_completa']).dt.month
datas_unicas['dia'] = pd.to_datetime(datas_unicas['data_completa']).dt.day
datas_unicas['trimestre'] = pd.to_datetime(datas_unicas['data_completa']).dt.quarter

dim_tempo = datas_unicas.reset_index().rename(columns={'index': 'id_tempo_sk'})
print(f"Dimensão Tempo: {len(dim_tempo)} registros")

# 2. Criando a Dimensão Ano (baseada no ano de referência)
if 'ano_referencia' in df_clean.columns:
    anos_unicos = df_clean[['ano_referencia']].dropna().drop_duplicates().sort_values('ano_referencia')
    anos_unicos['decada'] = (anos_unicos['ano_referencia'] // 10) * 10
    anos_unicos['periodo_escolar'] = anos_unicos['ano_referencia'].map(lambda x: f"Ano Letivo {x}")
else:
    # Cria anos padrão se não houver coluna
    anos_unicos = pd.DataFrame({
        'ano_referencia': [2022, 2023, 2024],
        'decada': [2020, 2020, 2020],
        'periodo_escolar': ['Ano Letivo 2022', 'Ano Letivo 2023', 'Ano Letivo 2024']
    })

dim_ano = anos_unicos.reset_index().rename(columns={'index': 'id_ano_sk'})
print(f"Dimensão Ano: {len(dim_ano)} registros")

# 3. Criando dimensão com características categóricas disponíveis
# Identifica colunas categóricas (strings com valores repetidos)
categorical_cols = []
exclude_cols = ['id', 'data_processamento', 'ano_referencia']

for col in df_clean.columns:
    if col not in exclude_cols and df_clean[col].dtype == 'object':
        if df_clean[col].nunique() < len(df_clean) * 0.8:  # Menos de 80% de valores únicos
            categorical_cols.append(col)

if categorical_cols:
    dim_caracteristicas = df_clean[categorical_cols].drop_duplicates().reset_index(drop=True)
    dim_caracteristicas.insert(0, 'id_caracteristicas_sk', range(1, len(dim_caracteristicas) + 1))
else:
    # Dimensão padrão se não houver características categóricas
    dim_caracteristicas = pd.DataFrame({
        'id_caracteristicas_sk': [1],
        'categoria_geral': ['GERAL'],
        'descricao': ['Categoria padrão']
    })

print(f"Dimensão Características: {len(dim_caracteristicas)} registros")
print(f"Características incluídas: {categorical_cols}")

print("\n✓ Dimensões criadas com sucesso!")

In [ ]:
# Criando a Tabela Fato
fato = df_clean.copy()

# Faz merge com dimensão ano se possível
if 'ano_referencia' in df_clean.columns:
    fato = fato.merge(dim_ano[['id_ano_sk', 'ano_referencia']], on='ano_referencia', how='left')
else:
    fato['id_ano_sk'] = 1

# Faz merge com dimensão características se possível
if categorical_cols and all(col in fato.columns for col in categorical_cols):
    fato = fato.merge(dim_caracteristicas, on=categorical_cols, how='left')
else:
    fato['id_caracteristicas_sk'] = 1

# Adiciona chave da dimensão tempo
fato['id_tempo_sk'] = 1

# Seleciona colunas para a tabela fato
fact_columns = ['id_tempo_sk', 'id_ano_sk']
if 'id_caracteristicas_sk' in fato.columns:
    fact_columns.append('id_caracteristicas_sk')

# Adiciona métricas numéricas se existirem
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if col not in ['id', 'ano_referencia'] and col in fato.columns:
        fact_columns.append(col)

# Se não houver métricas numéricas, adiciona contador
if len([col for col in fact_columns if col not in ['id_tempo_sk', 'id_ano_sk', 'id_caracteristicas_sk']]) == 0:
    fato['quantidade_registros'] = 1
    fact_columns.append('quantidade_registros')

fato_situacao_alunos = fato[fact_columns].copy()
fato_situacao_alunos = fato_situacao_alunos.drop_duplicates().reset_index(drop=True)
fato_situacao_alunos.insert(0, 'id_fato', range(1, len(fato_situacao_alunos) + 1))

print(f"Tabela Fato: {len(fato_situacao_alunos)} registros")
print("✓ Esquema estrela criado com sucesso!")

In [ ]:
# Configuração da conexão com banco de dados
usuario_db = 'postgres'               
senha_db = 'password'                
host_db = 'localhost'                    
porta_db = '5432'                    
nome_banco_db = 'escolas_dw'          

# Criando a connection string
DATABASE_URL = f'postgresql://{usuario_db}:{senha_db}@{host_db}:{porta_db}/{nome_banco_db}?client_encoding=utf8'
print("Configuração do banco:")
print(f"Host: {host_db}")
print(f"Banco: {nome_banco_db}")
print(f"Usuário: {usuario_db}")
print("\n⚠️ Ajuste as credenciais antes de executar a carga!")

In [ ]:
try:
    engine = create_engine(DATABASE_URL)
    print("Tentando conectar ao banco de dados...")
    
    # Testa a conexão
    with engine.connect() as conn:
        result = conn.execute("SELECT 1")
        result.fetchone()
    
    print("✓ Conexão estabelecida! Iniciando a carga do Esquema Estrela...")
    
    # Carrega as dimensões primeiro
    print("Carregando Dimensão Tempo...")
    dim_tempo.to_sql('dim_tempo', con=engine, if_exists='replace', index=False)
    
    print("Carregando Dimensão Ano...")
    dim_ano.to_sql('dim_ano', con=engine, if_exists='replace', index=False)
    
    print("Carregando Dimensão Características...")
    dim_caracteristicas.to_sql('dim_caracteristicas', con=engine, if_exists='replace', index=False)
    
    # Carrega a tabela fato por último
    print("Carregando Tabela Fato...")
    fato_situacao_alunos.to_sql('fato_situacao_alunos', con=engine, if_exists='replace', index=False)
    
    print("\nCARGA CONCLUÍDA! Todas as tabelas do Esquema Estrela foram criadas.")
    print(f"- dim_tempo: {len(dim_tempo)} registros")
    print(f"- dim_ano: {len(dim_ano)} registros") 
    print(f"- dim_caracteristicas: {len(dim_caracteristicas)} registros")
    print(f"- fato_situacao_alunos: {len(fato_situacao_alunos)} registros")

except Exception as e:
    print(f"✗ Erro ao conectar/carregar dados: {e}")
    print("\nVerifique se:")
    print("1. O PostgreSQL está rodando")
    print("2. O banco de dados existe") 
    print("3. As credenciais estão corretas")
    print("4. O usuário tem permissões adequadas")
    
    # Salva localmente como backup
    print("\nSalvando dados localmente...")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    dim_tempo.to_csv(f"data/final/dim_tempo_{timestamp}.csv", index=False)
    dim_ano.to_csv(f"data/final/dim_ano_{timestamp}.csv", index=False)
    dim_caracteristicas.to_csv(f"data/final/dim_caracteristicas_{timestamp}.csv", index=False)
    fato_situacao_alunos.to_csv(f"data/final/fato_situacao_alunos_{timestamp}.csv", index=False)
    
    print("✓ Dados salvos localmente em data/final/")

## Resumo Final

**Pipeline ETL executado com sucesso!**

### Estatísticas:
- **Dados extraídos**: Portal de Dados Abertos do Recife (2022-2024)
- **Registros processados**: Dados limpos e padronizados
- **Esquema dimensional**: 4 tabelas (3 dimensões + 1 fato)

### Próximos passos:
1. Configurar conexão com PostgreSQL